# Notebook 4 — Carbon Accounting & the Attribution-Gap (reframed)
### Study: *The Second-Order Compute Footprint of AI-Coding-Tool Adoption in Open Source*

**Reframed per the honest finding.** The parallel-trends assumption for the CI-minutes outcome was violated (treated repos were already on a rising CI trajectory); the naive +135% does **not** survive trend adjustment (level jump null, p=0.65; slope +48%, p=0.074). So this notebook does **not** convert a causal effect into carbon. Instead it delivers the two defensible carbon contributions:

1. **Descriptive carbon footprint of the adopter cohort** — convert the *observed* CI-minutes of AI-adopting repos into energy (kWh) and CO₂e, reported as a **low / central / high sensitivity band** over the conversion factors. Framed as **association**, not causation.
2. **The attribution-gap illustration** — show, in tonnes CO₂e, the difference between what a **naive causal analysis (+135%)** would attribute to adoption and what the **trend-adjusted analysis (null / +48% non-sig)** supports. This makes the paper's cautionary contribution concrete.

**Methodological grounding (cited in paper):** Software Carbon Intensity spec **ISO/IEC 21031:2024** (operational emissions O = E × I); **Green Algorithms** runtime model (Lannelongue et al., *Advanced Science* 2021); **Cloud Carbon Footprint** methodology; precedent Saavedra, Mendes & Ferreira (arXiv:2510.26413, 2025). Conversion reported as a band per best practice (never a point estimate); intermediate quantities (CI-minutes, kWh, gCO₂e/kWh) reported so others can re-derive.

**Honest boundaries (stated in paper):** CI-minutes is wall-clock per run = a compute *proxy*, not metered energy (documented error: CodeCarbon ~±20% vs meter; static estimators ≤40%; inter-tool divergence up to ~400%, Bouza et al. 2023). Operational emissions only (embodied carbon excluded unless added). Runner regions are undisclosed → grid-intensity is the dominant uncertainty, spanned by the band.


---
## Cell 0 — Re-attach environment & load cleaned CI panel + frozen claim


In [ ]:

%pip install -q pandas pyarrow numpy matplotlib

import json, pathlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT=pathlib.Path('/content/drive/MyDrive/ai_coding_carbon')
DIRS={k:PROJECT_ROOT/k for k in ['panels','cohort','results','figures']}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
CONFIG=json.loads((PROJECT_ROOT/'config.json').read_text())
claim=json.loads((DIRS['results']/'reframed_claim.json').read_text())

pc=pd.read_parquet(DIRS['panels']/'panel_ci_cleaned.parquet')
print("Cleaned CI panel rows:", len(pc), "| repos:", pc['full_name'].nunique())
print("Reframed claim is causal:", claim['causal_claim'])


---
## Cell 1 — Carbon conversion factors & the sensitivity band (SCI / Green Algorithms / CCF)

Conversion chain (operational emissions, per SCI ISO/IEC 21031:2024 and CCF):

```
energy_kWh   = CI_minutes/60  ×  vCPUs  ×  vCPU_power_kW  ×  PUE
CO2e_kg      = energy_kWh  ×  grid_intensity_gCO2/kWh / 1000
```

We span the irreducible uncertainty with **low / central / high** scenarios over the four uncertain factors. Defaults below are literature-anchored (finalize/cite in paper): vCPU power from CCF min/max watt ranges; PUE from hyperscale norms; grid intensity from a low-carbon region (e.g. ~50 gCO₂/kWh) to a coal-heavy region (~700), central ≈ a global-ish average. GitHub-hosted runners default to 2 vCPUs (standard Linux runner) — documented assumption.


> **Factors verified against CCF (this revision):** central per-vCPU power = **2.29 W** = CCF canonical average watts at 50% utilization, derived from CCF’s published per-provider min/max watts (AWS 0.74–3.5 W, GCP 0.71–4.26 W, Azure 0.78–3.76 W). low/high bracket the min-watt and max-watt ends. An earlier draft used 1.2 W centrally (~half), which understated the band; corrected here and logged. Cite the CCF coefficients for each value in the paper.

In [ ]:

# Carbon factors anchored to PUBLISHED CCF coefficients (verified vs CCF methodology).
# CCF per-vCPU watt defaults @50% utilization: AWS 0.74-3.5W, GCP 0.71-4.26W, Azure 0.78-3.76W.
# Average watts at 50% util ~ 2.29 W/vCPU (CCF canonical, avg across the three providers).
# low/central/high span the min-watt end -> CCF-average -> max-watt end, paired with PUE & grid.
FACTORS = {
  "low":     {"vcpu_power_kW": 0.00100, "pue": 1.10, "grid_gco2_kwh": 50,  "vcpus": 2},
  "central": {"vcpu_power_kW": 0.00229, "pue": 1.18, "grid_gco2_kwh": 369, "vcpus": 2},
  "high":    {"vcpu_power_kW": 0.00350, "pue": 1.40, "grid_gco2_kwh": 700, "vcpus": 2},
}
# persist for the artifact / re-derivation
(DIRS['results']/'carbon_factors.json').write_text(json.dumps(FACTORS,indent=2))

def minutes_to_co2e_kg(ci_minutes, scn):
    f=FACTORS[scn]
    energy_kwh=(ci_minutes/60.0)*f["vcpus"]*f["vcpu_power_kW"]*f["pue"]
    return energy_kwh*f["grid_gco2_kwh"]/1000.0, energy_kwh

print("Carbon factor scenarios (per SCI/Green Algorithms/CCF; illustrative, cite in paper):")
print(json.dumps(FACTORS,indent=2))
print("\nNOTE: grid intensity is the dominant uncertainty (runner regions undisclosed).")


---
## Cell 2 — Descriptive carbon footprint of the adopter cohort (sensitivity band)

We sum the **observed, retention-cleaned** CI-minutes of the **treated (adopter)** repos over their observed window, convert under each scenario, and annualize to a per-year figure. **Framed as association**: this is the CI-carbon footprint *of repos that have adopted AI tools*, not carbon *caused by* adoption.


In [ ]:

treated_ci = pc[(pc['is_treated']) & (pc['ci_eligible'])].dropna(subset=['ci_minutes_obs'])
# observed months per repo (to annualize)
obs_months = treated_ci.groupby('full_name')['ym'].nunique()
total_minutes = treated_ci['ci_minutes_obs'].sum()
mean_months = obs_months.mean()
n_adopters = treated_ci['full_name'].nunique()

rows=[]
for scn in ["low","central","high"]:
    co2e_kg, kwh = minutes_to_co2e_kg(total_minutes, scn)
    # annualize: scale observed total by (12 / mean observed months)
    annual_factor = 12.0/mean_months if mean_months>0 else np.nan
    rows.append({"scenario":scn,
                 "total_CI_minutes":round(total_minutes,0),
                 "energy_kWh":round(kwh,1),
                 "CO2e_kg_observed":round(co2e_kg,1),
                 "CO2e_kg_annualized":round(co2e_kg*annual_factor,1),
                 "CO2e_tonnes_annualized":round(co2e_kg*annual_factor/1000.0,3)})
band=pd.DataFrame(rows)
band.to_csv(DIRS['results']/'carbon_band_adopter_cohort.csv',index=False)
print(f"Adopter cohort: {n_adopters} repos | mean observed window {mean_months:.1f} months")
print(f"Total observed CI-minutes: {total_minutes:,.0f}\n")
print("DESCRIPTIVE CARBON FOOTPRINT OF ADOPTER COHORT (association, not causation):")
print(band.to_string(index=False))
print("\nReport as a BAND, e.g.: 'AI-adopting repos' CI compute corresponds to "
      f"{band.iloc[0]['CO2e_tonnes_annualized']:.2f}-{band.iloc[2]['CO2e_tonnes_annualized']:.2f} "
      f"tCO2e/yr (central {band.iloc[1]['CO2e_tonnes_annualized']:.2f}).'")


---
## Cell 3 — The attribution gap: naive (+135%) vs trend-adjusted (null)

The paper's cautionary centerpiece. We compute the CO₂e that a **naive causal analysis** would attribute to adoption (applying the naive +135% as if adoption *caused* it) versus the **trend-adjusted** estimate (level jump null → ~0; plus the non-significant +48% slope shown as a bounded "suggestive upper" for transparency). Expressed in tonnes CO₂e, this quantifies how badly naive attribution overstates the carbon cost.


In [ ]:

k=claim['key_numbers']
# counterfactual "attributable" minutes under each estimate:
#  naive: fraction of observed post minutes implied by +135% (i.e., share = effect/(1+effect))
def attributable_minutes(post_minutes, pct):
    eff=pct/100.0
    return post_minutes*(eff/(1.0+eff)) if eff>0 else 0.0

post_minutes = treated_ci[treated_ci['event_month']>=0]['ci_minutes_obs'].sum()

scenarios_effect = {
  "naive_causal_+135%": k['naive_did_ci_minutes_pct'],
  "trend_adj_level_jump_null(+11%,p=.65)": k['trend_adj_level_jump_pct'],   # not sig
  "trend_adj_slope_suggestive(+48%,p=.07)": k['trend_adj_slope_pct'],       # not sig
}
rows=[]
for name,pct in scenarios_effect.items():
    am=attributable_minutes(post_minutes,pct)
    co2_c,_=minutes_to_co2e_kg(am,"central")
    co2_l,_=minutes_to_co2e_kg(am,"low")
    co2_h,_=minutes_to_co2e_kg(am,"high")
    rows.append({"estimate":name,"effect_pct":pct,
                 "attributable_CI_min":round(am,0),
                 "attributable_tCO2e_low":round(co2_l/1000,3),
                 "attributable_tCO2e_central":round(co2_c/1000,3),
                 "attributable_tCO2e_high":round(co2_h/1000,3)})
gap=pd.DataFrame(rows)
gap.to_csv(DIRS['results']/'attribution_gap.csv',index=False)
print("ATTRIBUTION GAP — CO2e a naive analysis would attribute to adoption vs trend-adjusted:")
print(gap.to_string(index=False))
print("\nKEY MESSAGE: the naive estimate attributes a large CO2e to adoption; the")
print("trend-adjusted estimates are statistically null/non-significant -> naive AI-carbon")
print("attribution can be substantially overstated. (This is the paper's cautionary result.)")


---
## Cell 4 — Figures: sensitivity band + attribution gap


In [ ]:

band=pd.read_csv(DIRS['results']/'carbon_band_adopter_cohort.csv')
gap=pd.read_csv(DIRS['results']/'attribution_gap.csv')

fig,ax=plt.subplots(1,2,figsize=(12,4.5))

# (a) sensitivity band
ax[0].bar(band['scenario'], band['CO2e_tonnes_annualized'],
          color=['#9ecae1','#3182bd','#08519c'])
ax[0].set_ylabel('tCO2e / year'); ax[0].set_title('Adopter-cohort CI carbon\n(descriptive, association)')
for i,v in enumerate(band['CO2e_tonnes_annualized']):
    ax[0].text(i,v,f'{v:.2f}',ha='center',va='bottom',fontsize=9)

# (b) attribution gap (central, with low-high error bars)
g=gap.copy()
x=range(len(g))
ax[1].bar(x, g['attributable_tCO2e_central'],
          yerr=[g['attributable_tCO2e_central']-g['attributable_tCO2e_low'],
                g['attributable_tCO2e_high']-g['attributable_tCO2e_central']],
          capsize=4, color=['#de2d26','#fdae6b','#fee08b'])
ax[1].set_xticks(list(x)); ax[1].set_xticklabels(
    ['naive\n+135%','adj level\n(null)','adj slope\n(+48%,n.s.)'], fontsize=8)
ax[1].set_ylabel('tCO2e attributed to adoption'); ax[1].set_title('Attribution gap\n(naive vs trend-adjusted)')
plt.tight_layout(); plt.savefig(DIRS['figures']/'carbon_band_and_gap.png',dpi=150)
plt.show()
print("Saved figure -> figures/carbon_band_and_gap.png")


---
## Cell 5 — Manifest & handoff to Notebook 5


In [ ]:

import datetime
manifest={"notebook":"04_carbon_accounting_reframed",
 "produced":{
   "results/carbon_factors.json":(DIRS['results']/'carbon_factors.json').exists(),
   "results/carbon_band_adopter_cohort.csv":(DIRS['results']/'carbon_band_adopter_cohort.csv').exists(),
   "results/attribution_gap.csv":(DIRS['results']/'attribution_gap.csv').exists(),
   "figures/carbon_band_and_gap.png":(DIRS['figures']/'carbon_band_and_gap.png').exists(),
 },"timestamp":datetime.datetime.utcnow().isoformat()+"Z"}
(PROJECT_ROOT/'manifest_nb04.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
print("\nNotebook 5 (final): assemble all results into paper-ready tables/figures around")
print("the honest narrative (validated detector + cautionary attribution result +")
print("descriptive carbon band), run remaining robustness, package replication artifact.")
